# Classical Simulation via Stabiliser State Decomposition

The initial groundwork for utilising ZX-Calculus for classical simulation via stabiliser decomposition comes from this 2021 paper by Kissinger and van de Wetering: https://arxiv.org/abs/2109.01076. Since then, there has been much literature on this topic, with new decompositions found as well as new heuristics and strategies for applying the decompositions already known. At the end of this notebook, a list of relevant publications to date is included, and for an explanation of the core concepts and review of the literature see section 2.3 (and in particular section 2.3.4) of https://ora.ox.ac.uk/objects/uuid:54d6e7f4-b845-4275-98cd-cf6749530d9b/files/d9s1616869.

This notebook aims to demonstrate the high-level API in PyZX for classically simulating circuits with stabiliser decomposition. For a deeper explanation of the specific methods available, it is advised to read the publications associated with the decompositions and strategies of interest (or to examine the relevant py file in pyzx.simulation.decompositions or pyzx.simulation.strategies), which are listed in the metadata and may be retrieved as shown later in this notebook. Moreover, this notebook primarily focuses on strong simulation (as this is more thoroughly researched and implemented with ZX-Calculus).

In [1]:
import pyzx as zx
from pyzx.simulation import Decomp, Strategy

### Applying Decompositions

The following shows a minimal example of how to apply a stabiliser state decomposition to a ZX-diagram:

In [2]:
# Generate very simple example graph consisting of two connected Z-spiders:
g = zx.Graph()
g.add_vertex(zx.VertexType.Z,0,0)
g.add_vertex(zx.VertexType.Z,0,4)
g.add_edge((0,1))
zx.draw(g,labels=True)

# Apply an instance of the edge cutting decomposition to edge (0,1) of graph g, with vertex type X:
terms = zx.simulation.apply_decomp(Decomp.CUT_EDGE, g=g, e=(0,1), ty=zx.VertexType.X)

# Draw the two branching terms:
print("decomposes to the sum of the following terms (scalar factors not shown but exist in the data structure of the graphs):")
for h in terms.graphs:
    zx.draw(h,labels=True)

decomposes to the sum of the following terms (scalar factors not shown but exist in the data structure of the graphs):


Another example, with a larger graph, follows. In this case, we apply the magic2 decomposition which, instead of a graph and an edge, takes a graph and a list of vertices as arguments.

In [3]:
# Generate a larger scalar Clifford+T ZX-diagram:
q = 10
d = 350
seed = 42
g = zx.generate.cliffordT(q,d,seed=seed)
g.apply_state("0"*q)
g.apply_effect("0"*q)
zx.draw(g,scale=20,labels=True)
zx.simplify.full_reduce(g)
g.normalize()
zx.draw(g,scale=20,labels=True)

In [4]:
# apply the magic2 decomposition to vertices 479 and 506 of graph g:
terms = zx.simulation.apply_decomp(Decomp.MAGIC_2,g,[479,506])

# display both branching terms:
for h in terms.graphs:
    zx.draw(h,scale=20,labels=True)

Or if we don't want to actually apply the decomposition here but just check if it would be a valid application, we can run:

In [5]:
zx.simulation.check_valid(Decomp.MAGIC_2,g,[479,506])

True

This validity check is also automatically run before the decomposition is applied when zx.simulation.apply_decomp(...) is called.

As with zx.simulation.apply_decomp, the specific function this runs under the hood is tied to the specific decomposition but should always return True if the specified application of the decomposition is valid. If instead the specified application is not valid, it should throw an appropriate error like so:

In [6]:
zx.simulation.check_valid(Decomp.MAGIC_2,g,[479,506,227])

AssertionError: Invalid application of magic2 decomposition. Expected 2 vertices but 3 were specified.

Both zx.simulation.apply_decomp and zx.simulation.check_valid may take the same arguments for a particular decomposition but these arguments will vary from one decomposition to another. The first argument is always the Decomp enum entry (of type zx.simulation.Decomp) and the second argument is always the graph to which the decomposition is to be applied. Most decompositions will also require one or more subsequent arguments, such as an edge or vertex or list of vertices, etc. If in doubt, check the specification of the decomposition you're using from its file under pyzx/simulation/decompositions.

### (Preparation for) Simulating Circuits

Circuits can be created or imported into PyZX in a number of ways, such as via QASM code with zx.Circuit.load("circuit.qasm") or directly in code:

In [7]:
qasm = """
OPENQASM 2.0;
include "qelib1.inc";

qreg q[2];
h q[0];
cx q[0], q[1];
t q[1];
"""

circ = zx.Circuit.from_qasm(qasm)
g = circ.to_graph()
zx.draw(circ)

Alternatively, various types of random circuits can be generated with the pyzx.generate module, such as follows:

In [8]:
# Generate a scalar Clifford+T ZX-diagram:
q = 10    # number of qubits
d = 250   # number of gates (~depth)
seed = 42 # seed for random generation
g = zx.generate.cliffordT(q,d,seed=seed)
zx.draw(g,scale=20)

To strongly simulate a circuit with stabiliser decomposition in PyZX, it must first be a *scalar* ZX-diagram, meaning it must have no open inputs or outputs (i.e. no boundary vertices). If the ZX-diagram is circuit-like then the inputs and outputs can be plugged with states and effects like so:

In [9]:
bitstr = "0011010101" # some output bitstring whose probability we want to determine for this graph
g.apply_state("0"*q)
g.apply_effect(bitstr)
zx.draw(g,scale=20,labels=True)

Here, bitstr="0011010101" is a particular output bitstring we want to check. Scroll to the right end of the graph and you'll see the 0's correspond to 0-phase X-spiders and the 1's pi-phase X-spiders. By plugging the output of the circuit in this way, we will be aiming to ultimately calculate *"what is the probability that this quantum circuit will produce the output 0011010101 if it were to be executed on a (noiseless) quantum computer?"* This is the essence of strong classical simulation.

Next, we simplify the graph with the ZX rewriting rules via full_reduce:

In [10]:
zx.simplify.full_reduce(g)
g.normalize()
zx.draw(g,scale=20,labels=True)

Note that this (and all) ZX-diagrams also have an associated complex scalar factor not displayed, which may be exposed like so:

In [11]:
print(g.scalar)

-22.63+22.63i = sqrt(2)^10exp(3/4ipi)


Or expressed as a complex number rather than the more structured scalar data structure of PyZX:

In [12]:
print(g.scalar.to_number())

(-22.627416997969533+22.627416997969537j)


In the case of a *Clifford* scalar ZX-diagram, full_reduce will contract the entire graph to nothing more than a complex scalar. But in general, when the circuit includes non-Clifford gates (spiders of phases beyond n*pi/2 for integer n) the rewriting rules alone are insufficient to fully reduce the graph to a scalar, as observed in the example above. It will nevertheless typically still reudce the number of non-Clifford gates to some extent, which is very helpful for classical simulation (see: https://arxiv.org/abs/2109.01076).

### Simulating Circuits

A non-Clifford scalar ZX-diagram can then be reduced to its scalar amplitude with the help of stabiliser decomposition:

In [13]:
%%time
# draw the post-full_reduce example graph from earlier:
zx.draw(g,scale=20)

# fully decompose the graph using the 'BSS' decomposition strategy of https://arxiv.org/abs/2109.01076:
terms = zx.simulation.full_decompose(Strategy.BSS,g)

CPU times: total: 422 ms
Wall time: 420 ms


Here, we selected the 'BSS' decomposition strategy (https://arxiv.org/abs/2109.01076) to fully decompose the graph into a list (assigned to the 'terms' variable) of empty scalar graphs (essentially just complex scalar factors) whose sum represents the amplitude of the original graph.

We can now sum these terms together to determine the amplitude:

In [19]:
sum(term.scalar.to_number() for term in terms) # calculate the amplitude by summing the scalar terms

(-0.052104823300124316-0.007448649412080078j)

Or we can fully decompose and return the scalar directly by using *pyzx.simulation.simulate* instead of *pyzx.simulation.full_decompose*:

In [ ]:
%%time
# fully decompose the graph using the 'BSS' decomposition strategy and sum the resulting terms to determine the overall amplitude:
amplitude = zx.simulation.simulate(Strategy.BSS,g)
print(amplitude)

(-0.052104823300124316-0.007448649412080078j)
CPU times: total: 406 ms
Wall time: 409 ms


Thus we have deduced the scalar amplitude that the plugged ZX-diagram g represents. Relating back to the initial circuit, this then tells us the probability amplitude *A* of this circuit given the inputs (|00...0>) and outputs (<0011010101|>) we specified.

As such, we can now answer our initial question of *"what is the probability that this quantum circuit will produce the output 0011010101 if it were to be executed on a (noiseless) quantum computer?"*, and thus achieve strong classical simulation, by taking the absolute square of this amplitude:

In [31]:
# determine the probability of measuring this particular output bitstring (0011010101) from this circuit:
prob = abs(amplitude)**2
print(prob)

0.0027703949892012585


And just like that, we've strongly classically simulated our circuit.

Moreover, given we know the initial T-count of the graph (after full_reduce) as well as the number of terms produced by full_decompose, we can quantify the *effective alpha* (see later section) of this decomposition strategy (at least for this particular graph):

In [32]:
import math

t = zx.tcount(g)           # get the T-count of graph g
n = len(terms)             # count the number of terms produced after fully decomposing with the BSS strategy
alpha_eff = math.log2(n)/t # Calculate the effective alpha for this strategy (as applied to this graph)

print("Initial T-count:",t)
print("#terms produced:",n)
print("Effective alpha:",alpha_eff)

Initial T-count: 19
#terms produced: 407
Effective alpha: 0.4562571044350657


Thus we see that by using ZX-simplification after every application of a decomposition, we achieve more efficient simulation scaling than would be achieved otherwise, as this measured effective alpha is lower than the theoretical ("naive") alpha of this decomposition:

In [33]:
zx.simulation.get_alpha(Decomp.BSS)

0.4678924870096

### Additional Tips and Tricks

The decompositions also have associated metadata which may be useful to the user. For instance, the naive 'alpha' coefficient of a decomposition quantifies how efficient it is at removing T-spiders. If no ZX-rewriting simplification is applied during decomposition and all t T-spiders in a graph are removed entirely by recursive applications of a particular decomposition, the number of final terms produced would be 2^{alpha*t}. This is essentially how alpha is defined for a decomposition, namely alpha:=log_2(num_terms)/(T-count). (See definition 17 of https://ora.ox.ac.uk/objects/uuid:54d6e7f4-b845-4275-98cd-cf6749530d9b/files/d9s1616869.)

As such, a lower alpha decomposition is better (at least naively). (In practice, applying higher alpha decompositions with thoughtful heuristics can lead to fewer terms overall, after ZX-simplification is taken into account, resulting in a lower "effective alpha". See: https://arxiv.org/abs/2403.10964)

In PyZX, as demonstrated above, the naive alpha coefficient of a decomposition can be retrieved like so:

In [34]:
zx.simulation.get_alpha(Decomp.MAGIC_5)

0.3962406251803

For most decompositions, relevant literature references can also be retrieved like so:

In [35]:
zx.simulation.get_reference(Decomp.CUT_VERTEX)

'https://arxiv.org/pdf/2403.10964, https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/codsi-thesis.pdf'

This also works for decomposition *strategies*:

In [36]:
zx.simulation.get_reference(Strategy.BSS)

'https://arxiv.org/abs/2109.01076'

The decompositions and strategies currently available in PyZX can be found under the pyzx.simulation.decompositions and pyzx.simulation.strategies folders respectively, or with IntelliSense on the pyzx.simulation.Decomp and pyzx.simulation.Strategy enums. Alternatively, the options may be listed thusly:

In [37]:
list(Decomp)

[<Decomp.BSS: 'bss'>,
 <Decomp.CAT_3: 'cat_3'>,
 <Decomp.CAT_4: 'cat_4'>,
 <Decomp.CAT_5: 'cat_5'>,
 <Decomp.CAT_6: 'cat_6'>,
 <Decomp.CAT_N: 'cat_n'>,
 <Decomp.MAGIC_5: 'magic_5'>,
 <Decomp.MAGIC_2: 'magic_2'>,
 <Decomp.CUT_EDGE: 'cut_edge'>,
 <Decomp.CUT_VERTEX: 'cut_vertex'>,
 <Decomp.CUT_WISHBONE: 'cut_wishbone'>]

In [38]:
list(Strategy)

[<Strategy.BSS: 'bss'>, <Strategy.CUT_RANDOM: 'cut_random'>]

### Related Literature / Further Reading

1.    **"Trading classical and quantum computational resources"** (https://arxiv.org/abs/1506.01396) - BSS decomp. (6 T-gates -> 7 Clifford terms: alpha~=0.47)

2.    **"Simulating quantum circuits with ZX-calculus reduced stabiliser decompositions"** (https://arxiv.org/abs/2109.01076) - The initial paper on utilising ZX for stab decomp classical sim. Specifically, translating BSS into ZX and applying full_reduce after each decomp. to further reduce T-count

3.    **"Classical simulation of quantum circuits with partial and graphical stabiliser decompositions"** (https://arxiv.org/abs/2202.09202) - Cats decomps (lower alpha, but structure specific) + Magic5 decomp (state of the art alpha~=0.396)

4.    **"Cutting-Edge Graphical Stabiliser Decompositions for Classical Simulation of Quantum Circuits"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/codsi-thesis.pdf) - Early cutting decomp work (chapters 3-4), plus some gate-by-gate weak simulation (chapter 8)

5.    **"Speedy Contraction of ZX Diagrams with Triangles via Stabiliser Decompositions"** (https://arxiv.org/abs/2307.01803) - Finding new decomps. with triangle and star operators

6.    **"Graphical Stabilizer Decompositions For Counting Problems"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/laakkonen-thesis.pdf) - Similar to above, finding new decomps. with H-boxes

7.    **"Classically Simulating Quantum Supremacy IQP Circuits through a Random Graph Approach"** (https://arxiv.org/abs/2212.08609) - Extending ideas from [4] and applying them to IQP circuits 

8.    **"Procedurally Optimised ZX-Diagram Cutting for Efficient T-Decomposition in Classical Simulation"** (https://arxiv.org/abs/2403.10964) - Using the cutting decomp from [4], demonstrating how analysing the circuit in advance to find good cuts can lead to more efficient term counts than using a lower alpha decomposition. (Foundational to the KH papers [19,20])

9.   **"Fast Classical Simulation of Quantum Circuits via Parametric Rewriting in the ZX-Calculus"** (https://arxiv.org/abs/2403.06777) - "ParamZX" (precursor to TSim [21]), showing that the rewriting rules can (fully for Cliffords and almost fully for Clifford+T) be preserved for (appropriately restricted) parameterised phases such that branching terms can be expressed as a single parameterised diagram, leading to a parametric scalar ideal for efficient GPU computation. Average speedup factor of 78x.

10.   **"Smarter k-Partitioning of ZX-Diagrams for Improved Quantum Circuit Simulation"** (https://arxiv.org/abs/2409.00828) - a hyrbid method combining stabiliser decomposition with tensor contraction via graph partitioning

11.   **"Efficient Heuristics for Classical Simulation of Quantum Circuits Using ZX-Calculus"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/ahmad-thesis.pdf) - Building off of [8], showing some new (low alpha but very structure specific and situational) decompositions derived from special cases of the cutting decomp.

12.   **"Dynamic T-decomposition for classical simulation of quantum circuits"** (https://arxiv.org/abs/2412.17182) - Paper from [11] MSc thesis

13.   **"Towards Faster Quantum Circuit Simulation Using Graph Decompositions, GNNs and Reinforcement Learning"** (https://openreview.net/pdf?id=54060pbCKY) - Using reinforcement learning rather than random choice when selecting groups of spiders with which to apply Magic5 achieves marginally better results.

14.   **"GNNs for Fast Classical Simulation of Quantum Circuits through Optimizing ZX-Graph Decompositions"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/philipps-thesis.pdf) - related to [13], an MSc thesis on how ML can be used to help optimise the decomp. routine

15.   **"Novel methods for classical simulation of quantum circuits via ZX-calculus"** (https://ora.ox.ac.uk/objects/uuid:54d6e7f4-b845-4275-98cd-cf6749530d9b/files/d9s1616869) - PhD thesis covering papers [8,9,10,12,13]

16.   **"A Rank-Width–Based Approach to Quantum Circuit Simulation via ZX-Calculus"** (https://www.cs.ox.ac.uk/people/aleks.kissinger/theses/kuyanov-thesis.pdf) - A rank-width approach, using ZX for a tensor contraction approach to simulation

17.   **"Efficient Classical Simulation of Low-Rank-Width Quantum Circuits Using ZX-Calculus"** (https://arxiv.org/pdf/2603.06764) - A paper based on the [16] MSc thesis

18.   **"Unifying Graph Measures and Stabilizer Decompositions for the Classical Simulation of Quantum Circuits"** (https://arxiv.org/pdf/2603.06377) - Another hybrid approach based on tensor contraction and stabiliser decomposition

19.   **"Cutting stabiliser decompositions of magic state cultivation with ZX-calculus"** (https://arxiv.org/abs/2509.01224) - Cutting decomp techniques realted to [8] applied to magic state cultivation circuits, resulting in hugely reduced term counts

20.   **"Simulating magic state cultivation with few Clifford terms"** (https://arxiv.org/abs/2509.08658) - Followup paper to [19]

21.   **"Tsim: Fast Universal Simulator for Quantum Error Correction"** (https://arxiv.org/abs/2604.01059) - "TSim" open source software applying the parameterised/GPU-parallelised work of [9] to quantum error correction

<sup><sub>Jupyter Notebook by @mjsutcliffe99</sub></sup>